# 04 — Export tweets with **text** + **summaries** (+ emotions)

Notebook `03` writes the emotion final CSV **without** `message_id` / `text` (privacy-style export).

This notebook builds the deliverable you actually want to open in Excel:

- original **tweet text**
- **Tweet_Summary** (same approved LLM summary prompt as in `03`)
- emotion / theme columns from `03` when available

**Output:** `data/output/summaries/tweets_with_text_summaries_and_emotions.csv`  
(+ `.xlsx` when openpyxl is available)

Run from the `notebooks/` folder. Summaries resume from `data/output/summaries/summary_resume_checkpoint.jsonl`.


## 0. Setup


In [ ]:
from __future__ import annotations

import ast
import gc
import json
import os
import re
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_INPUT = ROOT / "data" / "input"
DATA_LABELED = ROOT / "data" / "output" / "labeled"
DATA_SUMMARIES = ROOT / "data" / "output" / "summaries"
DATA_CHECKPOINTS = ROOT / "data" / "output" / "checkpoints"
for _d in (DATA_INPUT, DATA_LABELED, DATA_SUMMARIES, DATA_CHECKPOINTS):
    _d.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_INPUT / "sample_geotweets.csv"
SUMMARY_CKPT = DATA_SUMMARIES / "summary_resume_checkpoint.jsonl"
EXPORT_CSV = DATA_SUMMARIES / "tweets_with_text_summaries_and_emotions.csv"
EXPORT_XLSX = DATA_SUMMARIES / "tweets_with_text_summaries_and_emotions.xlsx"

EMOTION_FINAL = DATA_LABELED / "tweets_with_emotion_labels_final.csv"
LLAMA_CSV = DATA_LABELED / "tweets_llama_llm_labels.csv"

MAX_TWEETS: Optional[int] = None  # set e.g. 20 for a smoke test
RESUME = True
MAX_TWEET_CHARS = 500
SUMMARY_MAX_NEW_TOKENS = 100
CHECKPOINT_EVERY = 10

HF_TOKEN = os.environ.get("HF_TOKEN")

PREFERRED_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
FALLBACK_MODEL = "Qwen/Qwen2.5-3B-Instruct"

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH.resolve())
print("EXPORT_CSV:", EXPORT_CSV.resolve())
print("DEVICE:", DEVICE)
print("HF_TOKEN set:", bool(HF_TOKEN))


## 1. Load tweets (keep `message_id` + `text`)


In [ ]:
assert DATA_PATH.exists(), f"Missing: {DATA_PATH}"

# sample_geotweets.csv is tab-separated
raw = pd.read_csv(DATA_PATH, sep="\t", dtype=str, keep_default_na=False)
if "message_id" not in raw.columns:
    # fallback if someone saved it comma-separated
    raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)

required = ["message_id", "text"]
missing = [c for c in required if c not in raw.columns]
assert not missing, f"Missing columns: {missing}"

work = raw.copy()
work["message_id"] = work["message_id"].astype(str).str.strip()
work["text"] = work["text"].fillna("").astype(str).str.strip()
work = work[work["text"].str.len() > 0].copy()
work = work.drop_duplicates(subset=["message_id"], keep="first").reset_index(drop=True)

if MAX_TWEETS is not None:
    work = work.head(int(MAX_TWEETS)).copy()
    print(f"SMOKE TEST: {len(work)} tweets")

print("Working rows:", len(work))
print("Columns:", list(work.columns)[:12], "...")
display(work[["message_id", "place", "tweet_date", "text"]].head(3))


## 2. Approved summary prompt + helpers

Same summary prompt used in notebook `03`.


In [ ]:
SUMMARY_SYSTEM_PROMPT = (
    "You rewrite one social-media tweet as a short emotion-preserving summary.\n"
    "\n"
    "Goal:\n"
    "- Keep WHAT the tweet is about AND HOW the author feels.\n"
    "- A later emotion classifier must recover the same emotion from your summary alone.\n"
    "\n"
    "Rules:\n"
    '- Return ONLY valid JSON with keys: "Tweet_Summary" and "Primary_Emotion"\n'
    "- Tweet_Summary: 1-2 sentences, plain English\n"
    "- Primary_Emotion: exactly one of: fear, anger, grief/sadness, outrage, anxiety, "
    "helplessness, sympathy, community concern, resilience/hope, neutral/informational\n"
    "- Explicitly keep emotional tone words when present "
    "(angry, furious, mourning, scared, hopeful, sarcastic, etc.)\n"
    "- If sarcastic, bitter, or mocking, say that — do not flatten into a calm factual report\n"
    "- Preserve intensity: furious/outraged is not the same as frustrated or concerned\n"
    "- Use only the tweet text; do not invent facts, names, places, or events\n"
    "- Do not remove emotion just to sound polite or neutral\n"
    "- If unclear / mostly links/emojis, say that and use neutral/informational\n"
    "\n"
    "Good example:\n"
    '{"Tweet_Summary":"The author is outraged that officials focus on banning TikTok "
    'instead of stopping school shootings.","Primary_Emotion":"outrage"}'
    "\n"
    "Bad example (too flat — do NOT do this):\n"
    '{"Tweet_Summary":"The tweet discusses government priorities regarding TikTok "
    'and school shootings.","Primary_Emotion":"community concern"}'
)


def build_summary_user_prompt(tweet_text: str) -> str:
    text = (tweet_text or "")[:MAX_TWEET_CHARS]
    return f'tweet:\n"""\n{text}\n"""\nRespond with JSON only.'


def extract_json_object(text: str) -> Optional[dict]:
    if not text:
        return None
    text = text.strip()
    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.I).strip()
    text = re.sub(r"^```(?:json)?\s*|```\s*$", "", text, flags=re.I | re.M).strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return None
    candidate = m.group(0)
    for parser in (json.loads, ast.literal_eval):
        try:
            obj = parser(candidate)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    return None


def clean_summary(text: Any) -> str:
    s = re.sub(r"\s+", " ", str(text or "").strip())
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        s = s[1:-1].strip()
    return s[:400]


def validate_summary_record(obj: Optional[dict], raw_text: str = "") -> str:
    obj = obj or {}
    summary = clean_summary(
        obj.get("Tweet_Summary", obj.get("tweet_summary", obj.get("s", obj.get("summary", ""))))
    )
    if not summary and raw_text:
        summary = clean_summary(raw_text.split("\n")[0])
    return summary


def load_summary_checkpoint() -> Dict[str, dict]:
    done: Dict[str, dict] = {}
    if not SUMMARY_CKPT.exists():
        return done
    with SUMMARY_CKPT.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                mid = str(row.get("message_id", "")).strip()
                if mid and clean_summary(row.get("Tweet_Summary", "")):
                    done[mid] = row
            except Exception:
                continue
    print(f"Loaded {len(done)} summaries from checkpoint")
    return done


def append_summary_checkpoint(row: dict) -> None:
    with SUMMARY_CKPT.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


print("Summary prompt length:", len(SUMMARY_SYSTEM_PROMPT))
_demo = '{"Tweet_Summary":"The tweet expresses mourning after a school shooting."}'
assert "mourning" in validate_summary_record(extract_json_object(_demo)).lower()
print("Parser OK")


## 3. LLM summary pass (resume-safe)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

_llm_tokenizer = None
_llm_model = None
_llm_loaded_id: Optional[str] = None


def _hf_kwargs() -> dict:
    return {"token": HF_TOKEN} if HF_TOKEN else {}


def load_causal_lm(model_id: str):
    global _llm_tokenizer, _llm_model, _llm_loaded_id
    if _llm_loaded_id == model_id and _llm_model is not None:
        return _llm_tokenizer, _llm_model

    if _llm_model is not None:
        try:
            del _llm_model
            del _llm_tokenizer
        except Exception:
            pass
        gc.collect()

    print(f"Loading LLM: {model_id} on {DEVICE} ...")
    tok = AutoTokenizer.from_pretrained(model_id, **_hf_kwargs())
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE.type in {"cuda", "mps"} else torch.float32,
        **_hf_kwargs(),
    )
    model.to(DEVICE)
    model.eval()
    _llm_tokenizer, _llm_model, _llm_loaded_id = tok, model, model_id
    print("LLM ready:", model_id)
    return tok, model


def resolve_model_id() -> str:
    try:
        AutoTokenizer.from_pretrained(PREFERRED_MODEL, **_hf_kwargs())
        return PREFERRED_MODEL
    except Exception as e:
        print(f"Could not access {PREFERRED_MODEL}: {e}")
        print(f"Falling back to {FALLBACK_MODEL}")
        return FALLBACK_MODEL


def format_chat_prompt(tok, messages: List[dict]) -> str:
    if getattr(tok, "chat_template", None):
        try:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    parts = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        parts.append(f"<|start_header_id|>{role}<|end_header_id|>\n\n{content}<|eot_id|>")
    parts.append("<|start_header_id|>assistant<|end_header_id|>\n\n")
    return "".join(parts)


def generate_summary_for_tweet(model_id: str, tweet_text: str) -> Tuple[str, str]:
    tok, model = load_causal_lm(model_id)
    messages = [
        {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
        {"role": "user", "content": build_summary_user_prompt(tweet_text)},
    ]
    prompt = format_chat_prompt(tok, messages)
    inputs = tok(prompt, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(SUMMARY_MAX_NEW_TOKENS),
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    gen_ids = output_ids[0][inputs["input_ids"].shape[-1] :]
    raw_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    return validate_summary_record(extract_json_object(raw_text), raw_text), raw_text


def run_summaries(df: pd.DataFrame) -> Dict[str, str]:
    done_rows = load_summary_checkpoint() if RESUME else {}
    summaries = {mid: clean_summary(r.get("Tweet_Summary", "")) for mid, r in done_rows.items()}
    pending = df[~df["message_id"].astype(str).isin(summaries.keys())].copy()
    print(f"Already summarized: {len(summaries)} | remaining: {len(pending)}")

    if pending.empty:
        return summaries

    model_id = resolve_model_id()
    load_causal_lm(model_id)
    n_since = 0
    for _, row in tqdm(pending.iterrows(), total=len(pending), desc="summaries"):
        mid = str(row["message_id"])
        try:
            summary, raw = generate_summary_for_tweet(model_id, row["text"])
            err = ""
        except Exception as e:
            summary, raw, err = "", "", repr(e)
            print(f"ERROR {mid}: {err}")
        rec = {
            "message_id": mid,
            "model_id": model_id,
            "Tweet_Summary": summary,
            "raw_response": raw[:2000],
            "error": err,
        }
        append_summary_checkpoint(rec)
        summaries[mid] = summary
        n_since += 1
        if n_since % CHECKPOINT_EVERY == 0:
            print(f"  progress: {len(summaries)} summaries")
    return summaries


summaries = run_summaries(work)
print("Summaries ready:", sum(1 for s in summaries.values() if len(s) > 5), "/", len(work))


## 4. Join emotions from notebook `03` (if present)


In [ ]:
# Attach summaries onto full tweet rows
export_df = work.copy()
export_df["Tweet_Summary"] = export_df["message_id"].map(lambda m: summaries.get(str(m), ""))

# Prefer llama labels (has message_id) for emotion join; fall back to final CSV via date/lat/lon
emotion_cols_wanted = [
    "Final Emotion",
    "Final Emotion Confidence Percentage",
    "gun_violence_related_majority",
    "Loss / Grief Expression Percentage",
    "Recent Shooting Event Mentions Percentage",
    "Retaliation Language Percentage",
    "Social Cohesion Breakdown Percentage",
    "Trauma Exposure Percentage",
    "Loss to Aggression Transition Signal Percentage",
    "Emoticon Sentiment",
    "Emoticon Sentiment Confidence Percentage",
    "roberta_primary_emotion",
    "roberta_emotion_confidence_pct",
    "llama32__primary_emotion",
    "llama32__emotion_confidence_pct",
]

if LLAMA_CSV.exists() and EMOTION_FINAL.exists():
    llama = pd.read_csv(LLAMA_CSV, dtype=str, keep_default_na=False)
    final = pd.read_csv(EMOTION_FINAL, dtype=str, keep_default_na=False)
    bridge = llama[["message_id", "date", "latitude", "longitude"]].drop_duplicates("message_id")
    bridge["message_id"] = bridge["message_id"].astype(str)
    key = ["date", "latitude", "longitude"]
    keep_final = [c for c in emotion_cols_wanted if c in final.columns]
    final_u = final[key + keep_final].drop_duplicates(key, keep="first")
    emo = bridge.merge(final_u, on=key, how="left")
    # also keep a few llama-native fields if final missing
    for c in [
        "llama32__primary_emotion",
        "llama32__emotion_confidence_pct",
        "llama32__gun_violence_related",
    ]:
        if c in llama.columns and c not in emo.columns:
            emo = emo.merge(llama[["message_id", c]].drop_duplicates("message_id"), on="message_id", how="left")
    export_df = export_df.merge(emo.drop(columns=key, errors="ignore"), on="message_id", how="left")
    print("Joined emotion labels from notebook 03 outputs")
elif LLAMA_CSV.exists():
    llama = pd.read_csv(LLAMA_CSV, dtype=str, keep_default_na=False)
    llama["message_id"] = llama["message_id"].astype(str)
    keep = ["message_id"] + [c for c in llama.columns if c.startswith("llama32__")]
    export_df = export_df.merge(llama[keep].drop_duplicates("message_id"), on="message_id", how="left")
    print("Joined llama labels only (final emotion CSV missing)")
else:
    print("No emotion files found — exporting text + summaries only")

# Normalize text for Excel (one row = one tweet)
export_df["text"] = (
    export_df["text"].astype(str).str.replace("\r\n", " ", regex=False).str.replace("\n", " ", regex=False)
)
export_df["Tweet_Summary"] = (
    export_df["Tweet_Summary"].astype(str).str.replace("\r\n", " ", regex=False).str.replace("\n", " ", regex=False)
)

print("Export shape:", export_df.shape)
print("Has text:", "text" in export_df.columns)
print("Summaries filled:", int((export_df["Tweet_Summary"].str.len() > 5).sum()), "/", len(export_df))
display(export_df[["message_id", "place", "text", "Tweet_Summary"]].head(5))


## 5. Export (CSV + Excel) — **includes tweet text**


In [ ]:
front = [
    "message_id",
    "tweet_date",
    "date",
    "place",
    "user_location",
    "latitude",
    "longitude",
    "GPS",
    "matched_incident",
    "text",
    "Tweet_Summary",
]
emotion_prefer = [
    "Final Emotion",
    "Final Emotion Confidence Percentage",
    "gun_violence_related_majority",
    "Loss / Grief Expression Percentage",
    "Recent Shooting Event Mentions Percentage",
    "Retaliation Language Percentage",
    "Social Cohesion Breakdown Percentage",
    "Trauma Exposure Percentage",
    "Loss to Aggression Transition Signal Percentage",
    "Emoticon Sentiment",
    "Emoticon Sentiment Confidence Percentage",
    "roberta_primary_emotion",
    "roberta_emotion_confidence_pct",
    "llama32__primary_emotion",
    "llama32__emotion_confidence_pct",
]
extra_meta = [
    "tags",
    "tweet_lang",
    "source",
    "retweets",
    "tweet_favorites",
    "followers",
    "friends",
    "user_favorites",
    "user_id",
    "user_name",
    "geom",
    "data_source",
    "spatialerror",
]

ordered = []
for c in front + emotion_prefer + extra_meta:
    if c in export_df.columns and c not in ordered:
        ordered.append(c)
for c in export_df.columns:
    if c not in ordered:
        ordered.append(c)

final_out = export_df[ordered].copy()
final_out["message_id"] = final_out["message_id"].astype(str)

# CSV: BOM + apostrophe prefix so Excel keeps full message_id
csv_out = final_out.copy()
csv_out["message_id"] = "'" + csv_out["message_id"].astype(str)
csv_out.to_csv(EXPORT_CSV, index=False, encoding="utf-8-sig")
print("Wrote CSV:", EXPORT_CSV.resolve(), "| rows=", len(csv_out))

# Also refresh the lighter summaries-only file
summary_cols = [c for c in front if c in final_out.columns]
sum_csv = final_out[summary_cols].copy()
sum_csv["message_id"] = "'" + sum_csv["message_id"].astype(str)
sum_csv.to_csv(SUMMARY_ONLY_CSV, index=False, encoding="utf-8-sig")
print("Updated:", SUMMARY_ONLY_CSV.resolve())

try:
    with pd.ExcelWriter(EXPORT_XLSX, engine="openpyxl") as writer:
        final_out.to_excel(writer, index=False, sheet_name="tweets")
        ws = writer.sheets["tweets"]
        ws.freeze_panes = "A2"
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=1).number_format = "@"
            ws.cell(row=row, column=1).value = str(ws.cell(row=row, column=1).value)
    print("Wrote XLSX:", EXPORT_XLSX.resolve())
except Exception as e:
    print("XLSX skipped:", e)

display(final_out[["message_id", "place", "text", "Tweet_Summary"]].head(5))


## 6. QC


In [ ]:
assert "text" in final_out.columns, "text column missing"
assert "Tweet_Summary" in final_out.columns, "Tweet_Summary missing"
assert "message_id" in final_out.columns, "message_id missing"
assert len(final_out) == len(work), "row count mismatch"
n_sum = int((final_out["Tweet_Summary"].astype(str).str.len() > 5).sum())
assert n_sum == len(final_out), f"missing summaries: {len(final_out) - n_sum}"
assert EXPORT_CSV.exists()

print("QC PASS")
print(f"  rows           : {len(final_out)}")
print(f"  summaries filled: {n_sum}/{len(final_out)}")
print(f"  has tweet text : yes")
print(f"  CSV            : {EXPORT_CSV.resolve()}")
if EXPORT_XLSX.exists():
    print(f"  XLSX           : {EXPORT_XLSX.resolve()}")
if "Final Emotion" in final_out.columns:
    print("\nFinal Emotion counts:")
    display(final_out["Final Emotion"].value_counts().head(10))
